In [1]:
import numpy as np
import sys
sys.path.insert(0, '../src')
from cr3bp import integrate_with_stm, cr3bp_rhs, MU_EARTH_MOON
from scipy.integrate import solve_ivp

mu = MU_EARTH_MOON

# A reference initial condition (somewhere with interesting dynamics)
x0 = np.array([0.8, 0.0, 0.0, 0.3])
t_final = 2.0

# 1 & 2: integrate reference trajectory WITH the STM
sol_ref = integrate_with_stm(x0, (0, t_final), mu, t_eval=[t_final])
ref_final = sol_ref.y[:4, -1]            # final trajectory state
Phi = sol_ref.y[4:, -1].reshape(4, 4)    # the STM at t_final

print("STM Phi at t_final:")
print(Phi)
print()

# 3-6: test the STM prediction against finite differences, one component at a time
epsilon = 1e-6
print(f"{'component':>10} | {'max error':>12}")
print("-" * 28)
for j in range(4):
    # Perturb the j-th component of the initial state
    dx0 = np.zeros(4)
    dx0[j] = epsilon

    # Integrate the perturbed trajectory (state only, no STM needed)
    sol_pert = solve_ivp(cr3bp_rhs, (0, t_final), x0 + dx0, args=(mu,),
                         method='DOP853', rtol=1e-12, atol=1e-12, t_eval=[t_final])
    pert_final = sol_pert.y[:, -1]

    # Actual vs predicted difference at t_final
    dx_actual = pert_final - ref_final          # brute force
    dx_predicted = Phi @ dx0                     # STM prediction

    max_err = np.max(np.abs(dx_actual - dx_predicted))
    print(f"{j:>10} | {max_err:>12.2e}")

STM Phi at t_final:
[[ 14.89462423  -5.16584591   4.02154172   3.41148837]
 [-23.838405     6.82853873  -7.7752053   -3.4105434 ]
 [ 20.0039876   -7.87011655   4.94074683   5.11535551]
 [-23.41881243   8.00831153  -6.29484799  -5.57815454]]

 component |    max error
----------------------------
         0 |     2.06e-09
         1 |     2.23e-10
         2 |     2.70e-10
         3 |     4.08e-11
